# solveit-tex

A utility for creating LaTeX papers using SolveIT, to facilitate Human-Authored, Computer-Interrogated (HACI) writing workflows.

In [ ]:
#| export
import os, subprocess, json, re, sys
from pathlib import Path
from IPython.display import HTML, display

## Utilities called by other routines

In [ ]:
#| export
def get_private_url(path: str):
    "Get the private URL for a file on the solveit cloud instance"
    server = os.getenv('PRIVATE_DOMAIN')
    if not server: raise ValueError("PRIVATE_DOMAIN not set")
    path = os.path.abspath(path)
    return f"https://{server}.solve.it.com{path.replace('/app/data', '/static')}"

In [ ]:
#| export
def parse_figure(line: str):
    """Parse markdown figure syntax with multiple images on one line: ![alt1](img1.png) ![alt2](img2.png)\{width=45% #fig:label}
     Images on one line get grouped into a single figure, with the final caption and label being the one used for the group"""
    import re
    
    # Look for escaped attributes at the end: \{...}
    attrs = ''
    attr_match = re.search(r'\\\{([^}]*)\}\s*$', line)
    if attr_match:
        attrs = attr_match.group(1)
        line = line[:attr_match.start()]  # Remove the attributes part
    
    # Find all image patterns on the line
    pattern = r'!\[([^\]]*)\]\(([^)]+)\)'
    matches = re.findall(pattern, line.strip())
    
    if not matches: return None
    
    images = []
    caption = ""
    
    for i, (alt, path) in enumerate(matches):
        img = {'path': path.strip()}
        images.append(img)
        
        # Last image sets the caption and label
        if i == len(matches) - 1:
            caption = alt
    
    # Extract width and label from attributes
    width_m = re.search(r'width=([^\s#]+)', attrs)
    label_m = re.search(r'#fig:([^\s}]+)', attrs)
    width = width_m.group(1) if width_m else None
    label = label_m.group(1) if label_m else None
    
    # Apply width to all images if specified
    if width:
        for img in images:
            img['width'] = width
    
    # If caption is just a filename, treat as no caption
    if caption:
        path_basenames = [Path(p['path']).name for p in images]
        if caption in path_basenames or caption in [p['path'] for p in images]:
            caption = ""
    
    return {'caption': caption, 'images': images, 'label': label}

In [ ]:
# Test it
s = r'![](images/mnist_ring_op_k_filmr_family.png) ![MNIST ring op$^k$ compositionality - FILMR \& Matrix Op](images/mnist_ring_op_k_mat_family.png)\{#fig:op_k}'
parse_figure(s)

{'caption': 'MNIST ring op$^k$ compositionality - FILMR \\& Matrix Op',
 'images': [{'path': 'images/mnist_ring_op_k_filmr_family.png'},
  {'path': 'images/mnist_ring_op_k_mat_family.png'}],
 'label': 'op_k'}

In [ ]:
#| export
from pyskills import allow 

allow(export_ipynb_to_tex)
allow(current_to_pdf)
allow(compile_latex)

<function __main__.compile_latex(tex_file: str, cwd: str = '.')>

Yes, and the caption for the first pair would be "img2", And the caption for the second pair would be "img4", And none of them in this case would have any width parameters or figure labels specified

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly. So in that example:
- First figure: a.png + b.png, caption = "img2"
- Second figure: c.png + d.png, caption = "img4"

And if any of them had `{width=...}` or `{#fig:...}` attributes, those would apply to the individual subfigures (width) or the overall figure (label from the last image).

Does this approach work for what you need?

Sounds good. Yes. So I guess we're going to need to modify the parsing and the making and even the extract script, because the extract script is going to have to check to see if it's getting multiple figure lines in a row somehow.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right. So we need to:

1. Update `parse_figure()` to handle consecutive image lines as a group
2. Modify `extract_ipynb_to_tex()` to detect these groups and emit a single `\begin{figure}` with multiple `\subfigure` commands
3. Use the last image's caption and label for the overall figure

Let me look at the current `parse_figure()` function to see what needs to change:





<details class='tool-usage-details' markdown='1'>
<summary><code>bash(cmd=&quot;grep -A 30 \&quot;def parse_figure\&quot; /app/data/…&quot;)→&quot;    \&quot;def parse_figure(line: str):\\n\&quot;,\n  …&quot;</code></summary>

```json
{
  "id": "call_e20642e13add4c50a94f66bd",
  "server": false,
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "grep -A 30 \"def parse_figure\" /app/data/solveit-tex/solveit-tex.ipynb | head -40"
    }
  },
  "result": "    \"def parse_figure(line: str):\\n\",\n    \"    \\\"Parse markdown figure syntax: ![caption](path1,path2){width=X% #fig:label}\\\"\\n\",\n    \"    import re\\n\",\n    \"    m = re.match(r'^!\\\\[([^\\\\]]*)\\\\]\\\\(([^)]+)\\\\)(?:\\\\{([^}]*)\\\\})?$', line.strip())\\n\",\n    \"    if not m: return None\\n\",\n    \"    \\n\",\n    \"    caption = m.group(1)\\n\",\n    \"    paths = [p.strip() for p in m.group(2).split(',')]\\n\",\n    \"    attrs = m.group(3) or ''\\n\",\n    \"    \\n\",\n    \"    # If caption is just a filename (matches any of the paths), treat as no caption\\n\",\n    \"    if caption:\\n\",\n    \"        path_basenames = [Path(p).name for p in paths]\\n\",\n    \"        if caption in path_basenames or caption in paths:\\n\",\n    \"            caption = \\\"\\\"\\n\",\n    \"    \\n\",\n    \"    width_m = re.search(r'width=([^\\\\s#]+)', attrs)\\n\",\n    \"    label_m = re.search(r'#fig:([^\\\\s}]+)', attrs)\\n\",\n    \"    width = width_m.group(1) if width_m else None\\n\",\n    \"    label = label_m.group(1) if label_m else None\\n\",\n    \"    \\n\",\n    \"    images = [{'path': p, 'width': width} if width else {'path': p} for p in paths]\\n\",\n    \"    return {'caption': caption, 'images': images, 'label': label}\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"cfa96c4a\",\n   \"metadata\": {},\n   \"outputs\": ["
}
```

</details>











<details class='tool-usage-details' markdown='1'>
<summary><code>bash(cmd=&quot;grep -B 5 -A 20 \&quot;parse_figure\&quot; /app/data…&quot;)→&quot;    \&quot;def parse_figure(line: str):\\n\&quot;,\n  …&quot;</code></summary>

```json
{
  "id": "call_7043ca42527d4e5f80c3b4e5",
  "server": false,
  "call": {
    "function": "bash",
    "arguments": {
      "cmd": "grep -B 5 -A 20 \"parse_figure\" /app/data/solveit-tex/solveit-tex.ipynb | grep -A 20 \"figure\""
    }
  },
  "result": "    \"def parse_figure(line: str):\\n\",\n    \"    \\\"Parse markdown figure syntax: ![caption](path1,path2){width=X% #fig:label}\\\"\\n\",\n    \"    import re\\n\",\n    \"    m = re.match(r'^!\\\\[([^\\\\]]*)\\\\]\\\\(([^)]+)\\\\)(?:\\\\{([^}]*)\\\\})?$', line.strip())\\n\",\n    \"    if not m: return None\\n\",\n    \"    \\n\",\n    \"    caption = m.group(1)\\n\",\n    \"    paths = [p.strip() for p in m.group(2).split(',')]\\n\",\n    \"    attrs = m.group(3) or ''\\n\",\n    \"    \\n\",\n    \"    # If caption is just a filename (matches any of the paths), treat as no caption\\n\",\n    \"    if caption:\\n\",\n    \"        path_basenames = [Path(p).name for p in paths]\\n\",\n    \"        if caption in path_basenames or caption in paths:\\n\",\n    \"            caption = \\\"\\\"\\n\",\n    \"    \\n\",\n    \"    width_m = re.search(r'width=([^\\\\s#]+)', attrs)\\n\",\n    \"    label_m = re.search(r'#fig:([^\\\\s}]+)', attrs)\\n\",\n    \"    width = width_m.group(1) if width_m else None\\n\",\n    \"    label = label_m.group(1) if label_m else None\\n\",\n    \"    \\n\",\n--\n--\n    \"im_dict = parse_figure(s)\\n\",\n    \"im_dict\"\n   ]\n  },\n  {\n   \"cell_type\": \"markdown\",\n   \"id\": \"812a7982\",\n   \"metadata\": {\n    \"solveit_ai\": true\n   },\n   \"source\": [\n    \"Okay, one problem, the Markdown string like above doesn't actually parse in Markdown, so you don't actually see both images in the SolveIt dialogue.\\n\",\n    \"In fact, all you see is something that looks like an ordinary hyperlink, instead of two separate images. So maybe this is a bit of a design question. If we want to show multiple images in the Markdown, but combine them into a single figure in the LaTeX, what's the best way to do that?\\n\",\n    \"\\n\",\n    \"##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->\\n\",\n    \"\\n\",\n    \"Good question. The current comma-separated syntax doesn't render in standard Markdown, so you're right that we need a different approach.\\n\",\n    \"\\n\",\n    \"One option is to use HTML `<figure>` tags, which Markdown supports:\\n\",\n    \"\\n\",\n    \"```html\\n\",\n--\n    \"            elif line.startswith('### '):\\n\",\n    \"                out.append(f'\\\\\\\\subsection{{{line[4:].strip()}}}\\\\n')\\n\",\n    \"            elif line.startswith('## '):\\n\",\n    \"                out.append(f'\\\\\\\\section{{{line[3:].strip()}}}\\\\n')\\n\",\n    \"            else:\\n\",\n    \"                fig_dict = parse_figure(line)\\n\",\n    \"                if fig_dict:\\n\",\n    \"                    out.append(make_figure(fig_dict))\\n\",\n    \"                else:\\n\",\n    \"                    out.append(line)\\n\",\n    \"\\n\",\n    \"            i += 1\\n\",\n    \"\\n\",\n    \"    final = '\\\\\\\\documentclass{article}\\\\n\\\\\\\\usepackage{graphicx}\\\\n\\\\n'\\n\",\n    \"    final += '\\\\n'.join(out) + '\\\\n\\\\n'\\n\",\n    \"    final += '\\\\\\\\end{document}\\\\n'\\n\",\n    \"    Path(output_path).write_text(final)\\n\",\n    \"    print(f'Created {output_path}')\\n\",\n    \"    output_url = get_private_url(output_path)\\n\",\n    \"    display(HTML(f'<a href=\\\"{output_url}\\\" target=\\\"_blank\\\">{output_url}</a>'))\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"70171d5c\","
}
```

</details>







I've looked at the current code structure. Here's what needs to change:

**Current state:**
- `parse_figure()` handles comma-separated paths on a single line
- The extraction script processes lines one at a time, calling `parse_figure()` on each

**What we need:**
1. **Modify `parse_figure()`** to parse a single image line (remove the comma-splitting logic)
2. **Add a grouping function** that scans consecutive lines and collects figure lines into groups
3. **Update the extraction logic** to:
   - Detect consecutive `![...](...){...}` lines
   - Group them into a single figure
   - Use the last image's caption and label for the overall figure
   - Each image keeps its own width parameter
   - Emit `\begin{figure}` with multiple `\includegraphics` commands (or `\subfigure` if we want sub-captions)

The key change is moving from line-by-line processing to a look-ahead approach where we check if the next line is also a figure before deciding whether to emit a figure or regular text.

Should I start implementing these changes?

Oh, one dumber thing we could do, which wouldn't require modifying the extract script. Any images that are supposed to go together as one figure could be on the same line.  That way, we don't have to modify the extract script or do anything fancy to it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So you're suggesting we just keep the current comma-separated syntax on a single line:

```markdown
![caption](img1.png,img2.png){width=45% #fig:label}
```

This would work for LaTeX extraction (no changes needed), but the SolveIt preview would show it as a broken link instead of actual images. We'd accept that trade-off for simplicity.

Is that what you mean?

No, no, no. It still has to parse like regular Markdown, remember.  Try again, what do you think I meant?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Oh! You mean put multiple standard Markdown image syntaxes on the same line:

```markdown
![alt1](img1.png){width=45%} ![alt2](img2.png)
![Actual Caption](img3.png){#fig:label}
```

Wait, but that's still two lines. You mean literally all on one line?

```markdown
![alt1](img1.png){width=45%} ![alt2](img2.png) ![Caption](img3.png){#fig:label}
```

This would render all three images inline in SolveIt, and the extraction script would just need to detect multiple image patterns on a single line and combine them into one `\begin{figure}` environment.

Precisely. One line in the source. The markdown renders normally. Our script doesn't have to keep track of multiple lines to decide what its state is going to be. Only the parser has to pay attention to whether it's got multiple images defined on a single line.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect. So we just need to update `parse_figure()` to use `re.findall()` to detect multiple `![caption](path){attrs}` patterns on a single line, then build the figure dict with all images, using the last one's caption and label.

Want me to implement that change?

Yes, go.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;def parse_figure(line: str):\n    \&quot;Parse …&quot;)→&quot;PermissionError: def/class not allowed: …&quot;</code></summary>

```json
{
  "id": "call_3e7f2c5dba7f4491b461e4ed",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "def parse_figure(line: str):\n    \"Parse markdown figure syntax with multiple images on one line: ![alt1](img1.png){width=45%} ![alt2](img2.png) ![Caption](img3.png){#fig:label}\"\n    import re\n    # Find all image patterns on the line\n    pattern = r'!\\[([^\\]]*)\\]\\(([^)]+)\\)(?:\\{([^}]*)\\})?'\n    matches = re.findall(pattern, line.strip())\n    \n    if not matches: return None\n    \n    images = []\n    caption = \"\"\n    label = None\n    \n    for i, (alt, path, attrs) in enumerate(matches):\n        attrs = attrs or ''\n        width_m = re.search(r'width=([^\\s#]+)', attrs)\n        label_m = re.search(r'#fig:([^\\s}]+)', attrs)\n        width = width_m.group(1) if width_m else None\n        \n        img = {'path': path.strip()}\n        if width: img['width'] = width\n        images.append(img)\n        \n        # Last image sets the caption and label\n        if i == len(matches) - 1:\n            caption = alt\n            label = label_m.group(1) if label_m else None\n    \n    # If caption is just a filename, treat as no caption\n    if caption:\n        path_basenames = [Path(p['path']).name for p in images]\n        if caption in path_basenames or caption in [p['path'] for p in images]:\n            caption = \"\"\n    \n    return {'caption': caption, 'images': images, 'label': label}\n\n# Test it\ns = '![alt1](images/img1.png){width=45%} ![alt2](images/img2.png) ![MNIST ring op$^k$ compositionality](images/img3.png){#fig:op_k}'\nparse_figure(s)"
    }
  },
  "result": "PermissionError: def/class not allowed: parse_figure. Code message has been added to dialog. Ask user to run it."
}
```

</details>







I wrote the updated `parse_figure()` function that uses `re.findall()` to detect multiple `![alt](path){attrs}` patterns on a single line. The code was added to the dialog as message #_24b491c2 - you'll need to run it to test.

The function:
- Finds all image patterns on the line
- Builds an images list where each can have its own width
- Uses the last image's caption and label for the overall figure
- Handles the filename-as-caption edge case

Once you confirm it works, we can integrate it into the actual dialog.

Okay, I'm having a problem. The first image is appearing in solve it, but the second one is just appearing as like a hyperlink string: 
```
![](images/mnist_ring_op_k_filmr_family.png) ![MNIST ring op$^k$ compositionality - FILMR \& Matrix Op](images/mnist_ring_op_k_mat_family.png){#fig:op_k}
```
What's wrong with my second Markdown image specification

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The `{#fig:op_k}` attribute syntax isn't standard Markdown - it's a Pandoc/Quarto extension. SolveIt's Markdown renderer probably doesn't recognize it, which is why the second image isn't rendering.

Try removing the attributes:

```markdown
![](images/mnist_ring_op_k_filmr_family.png) ![MNIST ring op$^k$ compositionality - FILMR \& Matrix Op](images/mnist_ring_op_k_mat_family.png)
```

Does that render both images?

yes...? But the extra attributes shouldn't be a problem normally, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right that in Pandoc/Quarto-flavored Markdown, `{#fig:op_k}` works fine. But SolveIt's renderer apparently doesn't support that attribute syntax after image links - when it encounters `{...}` after the `)`, it fails to recognize the whole thing as an image and just shows the raw text.

So the attributes work for LaTeX extraction, but break SolveIt's rendering. That's the core tension we need to solve.

well and if I remove the second caption...
```
![](images/mnist_ring_op_k_filmr_family.png) ![](images/mnist_ring_op_k_mat_family.png){#fig=blah}
```
Then it does actually render both images, and then under the second image is the text "`{#fig=blah}`"

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So SolveIt's renderer treats `{#fig=blah}` as literal text rather than as image attributes. It's not parsing that syntax at all.

This means we have a fundamental incompatibility: the `{...}` attribute syntax works for LaTeX extraction but breaks SolveIt's Markdown rendering.

We need either:
1. A different syntax that SolveIt recognizes for attributes
2. A way to hide attributes from SolveIt's renderer but still extract them
3. Accept that multi-image figures won't render properly in SolveIt

What do you think is the best path forward?

In [ ]:
#| export
def make_figure(fig_dict: dict):
    "Generate LaTeX figure environment from image specs."
    images, caption, label = fig_dict['images'],  fig_dict.get('caption', ''), fig_dict.get('label', '')
    lines = ['\\begin{figure}[htbp]', '\\centering']  # Start figure environment
    for img in images:
        width_opt = f'[width={img["width"]}]' if 'width' in img else '[width=\\linewidth]' # Add width if specified, defalt to linewidth
        lines.append(f'\\includegraphics{width_opt}{{{img["path"]}}}')  # Include the image
    if caption: lines.append(f'\\caption{{{caption}}}')  # Add caption if provided
    if label: lines.append(f'\\label{{fig:{label}}}')  # Add label if provided
    lines.append('\\end{figure}')  # Close figure environment
    return '\n'.join(lines)

In [ ]:
fig_tex = make_figure(im_dict)
fig_tex

'\\begin{figure}[htbp]\n\\centering\n\\includegraphics[width=\\linewidth]{images/mnist_ring_op_k_filmr_family.png}\n\\includegraphics[width=\\linewidth]{images/mnist_ring_op_k_mat_family.png}\n\\caption{MNIST ring op$^k$ compositionality - FILMR \\& Matrix Op}\n\\label{fig:op_k}\n\\end{figure}'

## Conversion script for .ipynb -> .tex

In [ ]:
#| export
def export_ipynb_to_tex(ipynb_path: str, output_path: str = None):
    r"""Export a Solveit dialog (.ipynb) to a compilable LaTeX file.
    Cells are emitted in document order, each preceded by a `% <cell-id>` comment.
    The `## Abstract` cell emits `\begin{document}`, dividing preamble from document body."""

    ipynb_path = os.path.expanduser(ipynb_path)
    output_path = os.path.expanduser(output_path) if output_path else Path(ipynb_path).with_suffix('.tex')

    nb = json.loads(Path(ipynb_path).read_text())
    out = []

    for cell in nb['cells']:
        content = ''.join(cell['source'])

        if '#| export' not in content:
            continue

        filtered = '\n'.join(l for l in content.split('\n') if not l.startswith('#| '))
        out.append(f'% {cell["id"]}')

        if cell['cell_type'] == 'raw':
            out.append(filtered)
            continue

        lines = filtered.split('\n')
        i = 0
        while i < len(lines):
            line = lines[i]

            if line.startswith('# ') and not line.startswith('## '):
                out.append(f'\\title{{{line[2:].strip()}}}\n')
            elif line.startswith('\\author{'):
                out.append(line)
                while i < len(lines) and not lines[i].strip().endswith('}'):
                    i += 1
                    if i < len(lines):
                        out.append(lines[i])
            elif line == '## Abstract':
                out.append('\\begin{document}\n\n\\maketitle\n')
                out.append('\\begin{abstract}\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    out.append(lines[i])
                    i += 1
                out.append('\\end{abstract}\n')
                continue
            elif line == '## References':
                out.append('\\small\n')
                bib_match = re.search(r'(\w+)\.bib', content)
                if bib_match:
                    out.append(f'\\bibliographystyle{{unsrt}}\n\\bibliography{{{bib_match.group(1)}}}\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    i += 1
                continue
            elif line.startswith('### '):
                out.append(f'\\subsection{{{line[4:].strip()}}}\n')
            elif line.startswith('## '):
                out.append(f'\\section{{{line[3:].strip()}}}\n')
            else:
                fig_dict = parse_figure(line)
                if fig_dict:
                    out.append(make_figure(fig_dict))
                else:
                    out.append(line)

            i += 1

    final = '\\documentclass{article}\n\\usepackage{graphicx}\n\n'
    final += '\n'.join(out) + '\n\n'
    final += '\\end{document}\n'
    Path(output_path).write_text(final)
    print(f'Created {output_path}')
    output_url = get_private_url(output_path)
    display(HTML(f'<a href="{output_url}" target="_blank">{output_url}</a>'))

In [ ]:
export_ipynb_to_tex('example/neurips_2026.ipynb')

Created example/neurips_2026.tex


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex</a>)

## Conversion script for tex -> pdf

In [ ]:
#| export
def compile_latex(tex_file: str, cwd: str = '.'):
    "Run full LaTeX compilation: pdflatex → bibtex → pdflatex → pdflatex"

    cwd = os.path.expanduser(cwd)
    tex_file = os.path.expanduser(tex_file)
    
    # Make path absolute before splitting
    full_tex_path = os.path.abspath(os.path.join(cwd, tex_file))
    work_dir = os.path.dirname(full_tex_path)
    
    base_name = os.path.splitext(os.path.basename(full_tex_path))[0]
    tex_content = Path(full_tex_path).read_text()
    bib_match = re.search(r'\\bibliography\{([^}]+)\}', tex_content)
    
    print("Running pdflatex (pass 1)...")
    r1 = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                   shell=True, cwd=work_dir, capture_output=True, text=True)
    if r1.returncode != 0:
        print(f"  FAILED:\n{r1.stdout[-3000:]}")
        return
    
    if bib_match:
        bib_name = bib_match.group(1)
        print(f"\nRunning bibtex ({bib_name})...")
        r = subprocess.run(f'bibtex {base_name}',
                          shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout}")
            return
    
    for i in [2, 3]:
        print(f"\nRunning pdflatex (pass {i})...")
        r = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                       shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout[-3000:]}")
            return

    pdf_path = full_tex_path.replace('.tex', '.pdf') 
    pdf_url = get_private_url(pdf_path)
    print(f"\nSuccess!")
    print(f"File: {pdf_path}")
    print(f"PDF url: {pdf_url}")
    sys.stdout.flush() 
    display(HTML(f'<a href="{pdf_url}" target="_blank">{pdf_url}</a>'))

In [ ]:
compile_latex('example/neurips_2026.tex')

Running pdflatex (pass 1)...



Running bibtex (references)...



Running pdflatex (pass 2)...



Running pdflatex (pass 3)...



Success!


File: /app/data/solveit-tex/example/neurips_2026.pdf


PDF url: https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf</a>)

## All-in-One Wrapper Function

In [ ]:
#| export
async def current_to_pdf():
    """
    Wrapper that converts the current dialogue to PDF and prints the private URL for it.
    Usage: await current_to_pdf()
    """
    from dialoghelper.solveitskill import curr_dialog, realpath

    name = (await curr_dialog())['name']
    path = f'{await realpath("/")}/{name}.ipynb'
    export_ipynb_to_tex(path)
    display(HTML(f'<br>'))
    compile_latex(path.replace('.ipynb', '.tex'))

## Allow LLM to call routines herein